# 🚗 Sistema de Navegación Autónoma — Algoritmo A*
### Empresa de Avances Tecnológicos Automovilísticos · Departamento de IA

---

**Proyecto:** Prototipo de Búsqueda de Rutas Óptimas  
**Algoritmo principal:** A* (A-Star)  
**Tecnología:** Python · NumPy · Matplotlib  
**Objetivo:** Evaluar y comparar algoritmos de búsqueda de caminos sobre un mapa en cuadrícula que representa áreas urbanas, con soporte para movimiento diagonal, zonas especiales de coste variable e interfaz interactiva.

---

## 📚 Fase de Investigación

### 1. ¿Qué es el Algoritmo A*?

A* (pronunciado "A estrella") es un algoritmo de búsqueda de caminos informado que combina las ventajas de **Dijkstra** (garantía de optimalidad) y la **Búsqueda Voraz** (velocidad mediante heurísticas). Fue publicado por Hart, Nilsson y Raphael en 1968 y sigue siendo el estándar de la industria en sistemas de navegación, videojuegos y robótica.

A* evalúa cada nodo con la función:

> **f(n) = g(n) + h(n)**

- **g(n)**: coste real acumulado desde el origen hasta el nodo *n*
- **h(n)**: estimación heurística del coste desde *n* hasta el destino
- **f(n)**: coste total estimado del camino que pasa por *n*

El algoritmo mantiene una **lista abierta** (nodos pendientes de explorar, ordenada por f) y una **lista cerrada** (nodos ya visitados).

---

### 2. Heurísticas

La calidad de A* depende de la función heurística. Para que A* sea **admisible** (nunca sobreestima el coste real), la heurística debe ser *consistente*.

| Heurística | Fórmula | Movimiento | Admisible |
|---|---|---|---|
| **Manhattan** | \|Δx\| + \|Δy\| | Solo cardinal | ✅ |
| **Euclídea** | √(Δx² + Δy²) | Diagonal | ✅ |
| **Octile** | max(Δx,Δy) + (√2−1)·min(Δx,Δy) | Diagonal (óptima) | ✅ |
| **Chebyshev** | max(Δx, Δy) | Diagonal uniforme | ✅ |

Para este sistema usamos **Octile**, ya que es la más precisa cuando el coste de movimiento diagonal es √2 ≈ 1.414.

---

### 3. Movimiento Diagonal y Prevención de "Corner Cutting"

Para permitir rutas en diagonal o zigzag evitamos el "corte de esquinas" (*corner cutting*): un agente no puede atravesar diagonalmente si ambas celdas adyacentes ortogonalmente están bloqueadas. Esto garantiza realismo físico.

---

### 4. Algoritmos Comparados

| Algoritmo | Tipo | Óptimo | Completo | Complejidad temporal |
|---|---|---|---|---|
| **A*** | Informado | ✅ (con h admisible) | ✅ | O(b^d) |
| **Dijkstra** | No informado | ✅ | ✅ | O((V+E)·log V) |
| **BFS** | No informado | ✅ (coste uniforme) | ✅ | O(V+E) |
| **Greedy BFS** | Informado | ❌ | ✅ | O(b^m) |

- *b*: factor de ramificación, *d*: profundidad solución, *m*: profundidad máxima, *V*: vértices, *E*: aristas

## ⚙️ Instalación e Importaciones

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORTACIONES — todas incluidas en Google Colab por defecto
# ─────────────────────────────────────────────────────────────
import heapq
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.gridspec import GridSpec
import time
from collections import deque
import warnings
warnings.filterwarnings('ignore')

print("✅ Todas las librerías cargadas correctamente.")
print(f"   NumPy  {np.__version__}  |  Matplotlib {plt.matplotlib.__version__}")

## 🗺️ Estructura del Mapa (Grid)

In [ ]:
# ═══════════════════════════════════════════════════════
# CONSTANTES — Tipos de celda y costes de terreno
# ═══════════════════════════════════════════════════════
EMPTY, WALL, START, END, PATH, VISITED, SLOW, FAST = range(8)

TERRAIN_COST = {
    EMPTY: 1.0,   # Calle normal
    SLOW:  3.0,   # Obras / atasco
    FAST:  0.5,   # Autopista / vía rápida
    START: 1.0,
    END:   1.0,
}

# Colores para la visualización
COLORS = ['#FFFFFF',  # EMPTY   — blanco
          '#2C2C2C',  # WALL    — gris oscuro
          '#00CC44',  # START   — verde
          '#FF4444',  # END     — rojo
          '#3399FF',  # PATH    — azul
          '#FFE066',  # VISITED — amarillo
          '#FF9933',  # SLOW    — naranja
          '#99FFCC']  # FAST    — verde claro

CMAP = ListedColormap(COLORS)
NORM = BoundaryNorm(range(9), CMAP.N)


# ═══════════════════════════════════════════════════════
# CLASE GRID — Representa el mapa de la ciudad
# ═══════════════════════════════════════════════════════
class Grid:
    """Cuadrícula que modela las áreas de una ciudad urbana."""

    def __init__(self, rows: int, cols: int):
        self.rows = rows
        self.cols = cols
        self.grid = np.zeros((rows, cols), dtype=int)
        self.start = None
        self.end = None

    # ── Modificar celdas ────────────────────────────
    def set_cell(self, row: int, col: int, cell_type: int):
        if not (0 <= row < self.rows and 0 <= col < self.cols):
            print(f"  ⚠️  Celda ({row},{col}) fuera del mapa ({self.rows}x{self.cols})")
            return False
        # Si hay un START/END anterior, restaurar a EMPTY
        if cell_type == START and self.start:
            self.grid[self.start] = EMPTY
        if cell_type == END and self.end:
            self.grid[self.end] = EMPTY
        self.grid[row][col] = cell_type
        if cell_type == START:
            self.start = (row, col)
        elif cell_type == END:
            self.end = (row, col)
        return True

    def set_walls_from_list(self, coords: list):
        """Bloquear múltiples celdas a la vez. coords = [(r,c), ...]"""
        for r, c in coords:
            self.set_cell(r, c, WALL)

    # ── Propiedades de celdas ───────────────────────
    def get_cost(self, row: int, col: int) -> float:
        return TERRAIN_COST.get(self.grid[row][col], 1.0)

    def is_walkable(self, row: int, col: int) -> bool:
        return (0 <= row < self.rows and
                0 <= col < self.cols and
                self.grid[row][col] != WALL)

    # ── Vecinos con soporte diagonal ─────────────────
    def get_neighbors(self, row: int, col: int, diagonal: bool = True):
        """
        Devuelve lista de ((nr, nc), move_cost).
        Movimiento cardinal: coste 1.0
        Movimiento diagonal: coste √2 ≈ 1.414 (sin corner cutting)
        """
        neighbors = []
        cardinal = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        diag     = [(-1,-1), (-1, 1), (1,-1), (1, 1)]

        for dr, dc in cardinal:
            nr, nc = row + dr, col + dc
            if self.is_walkable(nr, nc):
                neighbors.append(((nr, nc), 1.0))

        if diagonal:
            for dr, dc in diag:
                nr, nc = row + dr, col + dc
                # Prevención de corner cutting
                if (self.is_walkable(nr, nc) and
                        self.is_walkable(row + dr, col) and
                        self.is_walkable(row, col + dc)):
                    neighbors.append(((nr, nc), 1.414))

        return neighbors

    # ── Utilidades ──────────────────────────────────
    def copy(self):
        g = Grid(self.rows, self.cols)
        g.grid = self.grid.copy()
        g.start = self.start
        g.end   = self.end
        return g

    def reset_path(self):
        """Elimina marcas de ruta y nodos visitados."""
        mask = (self.grid == PATH) | (self.grid == VISITED)
        self.grid[mask] = EMPTY

    def path_cost(self, path: list) -> float:
        """Calcula el coste real de un camino."""
        if not path:
            return float('inf')
        cost = 0.0
        for i in range(1, len(path)):
            r0, c0 = path[i-1]
            r1, c1 = path[i]
            move_cost = 1.414 if (abs(r1-r0) == 1 and abs(c1-c0) == 1) else 1.0
            cost += move_cost * self.get_cost(r1, c1)
        return cost

    def __repr__(self):
        return f"Grid({self.rows}x{self.cols}, start={self.start}, end={self.end})"


print("✅ Clase Grid definida.")

## 📐 Funciones Heurísticas

In [ ]:
# ═══════════════════════════════════════════════════════
# HEURÍSTICAS
# ═══════════════════════════════════════════════════════

def h_manhattan(a, b):
    """Distancia Manhattan — solo movimiento cardinal."""
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def h_euclidean(a, b):
    """Distancia Euclídea."""
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2) ** 0.5

def h_octile(a, b):
    """
    Distancia Octile — óptima para movimiento diagonal con coste √2.
    Fórmula: max(dx,dy) + (√2 - 1)·min(dx,dy)
    """
    dx, dy = abs(a[0]-b[0]), abs(a[1]-b[1])
    return max(dx, dy) + 0.414 * min(dx, dy)

def h_chebyshev(a, b):
    """Distancia Chebyshev — diagonal con coste uniforme 1."""
    return max(abs(a[0]-b[0]), abs(a[1]-b[1]))

HEURISTICS = {
    'Octile'    : h_octile,
    'Manhattan' : h_manhattan,
    'Euclídea'  : h_euclidean,
    'Chebyshev' : h_chebyshev,
}

print("✅ Heurísticas definidas:", list(HEURISTICS.keys()))

## 🧠 Implementación de Algoritmos

In [ ]:
# ═══════════════════════════════════════════════════════
# ALGORITMO A*
# ═══════════════════════════════════════════════════════

def astar(grid: Grid, heuristic=h_octile, diagonal: bool = True):
    """
    Búsqueda A* con soporte para movimiento diagonal y terrenos de coste variable.

    Parámetros
    ----------
    grid      : Grid con start y end definidos
    heuristic : función h(a, b) → float
    diagonal  : permitir movimiento en 8 direcciones

    Retorna
    -------
    path           : lista de (row, col) o None si no hay ruta
    nodes_explored : número de nodos extraídos de la open list
    elapsed_ms     : tiempo de ejecución en milisegundos
    visited_set    : conjunto de nodos explorados (para visualización)
    """
    start, end = grid.start, grid.end
    if start is None or end is None:
        return None, 0, 0, set()

    # Cola de prioridad: (f, (r,c))
    open_heap = []
    heapq.heappush(open_heap, (0.0, start))

    came_from  = {}
    g_score    = {start: 0.0}
    visited    = set()
    nodes_exp  = 0
    t0         = time.perf_counter()

    while open_heap:
        f, current = heapq.heappop(open_heap)

        if current in visited:
            continue
        visited.add(current)
        nodes_exp += 1

        if current == end:
            # ── Reconstruir camino ──
            path = []
            node = current
            while node in came_from:
                path.append(node)
                node = came_from[node]
            path.append(start)
            path.reverse()
            elapsed_ms = (time.perf_counter() - t0) * 1000
            return path, nodes_exp, elapsed_ms, visited

        for (nr, nc), move_cost in grid.get_neighbors(*current, diagonal=diagonal):
            neighbor = (nr, nc)
            if neighbor in visited:
                continue
            terrain = grid.get_cost(nr, nc)
            new_g   = g_score[current] + move_cost * terrain
            if neighbor not in g_score or new_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor]   = new_g
                f_val = new_g + heuristic(neighbor, end)
                heapq.heappush(open_heap, (f_val, neighbor))

    elapsed_ms = (time.perf_counter() - t0) * 1000
    return None, nodes_exp, elapsed_ms, visited


# ═══════════════════════════════════════════════════════
# ALGORITMO DIJKSTRA
# ═══════════════════════════════════════════════════════

def dijkstra(grid: Grid, diagonal: bool = True):
    """Dijkstra — A* con heurística cero. Garantiza solución óptima."""
    start, end = grid.start, grid.end
    if start is None or end is None:
        return None, 0, 0, set()

    open_heap  = [(0.0, start)]
    came_from  = {}
    dist       = {start: 0.0}
    visited    = set()
    nodes_exp  = 0
    t0         = time.perf_counter()

    while open_heap:
        cost, current = heapq.heappop(open_heap)
        if current in visited:
            continue
        visited.add(current)
        nodes_exp += 1

        if current == end:
            path = []
            node = current
            while node in came_from:
                path.append(node)
                node = came_from[node]
            path.append(start)
            path.reverse()
            return path, nodes_exp, (time.perf_counter()-t0)*1000, visited

        for (nr, nc), move_cost in grid.get_neighbors(*current, diagonal=diagonal):
            neighbor    = (nr, nc)
            terrain     = grid.get_cost(nr, nc)
            new_cost    = cost + move_cost * terrain
            if neighbor not in dist or new_cost < dist[neighbor]:
                dist[neighbor]      = new_cost
                came_from[neighbor] = current
                heapq.heappush(open_heap, (new_cost, neighbor))

    return None, nodes_exp, (time.perf_counter()-t0)*1000, visited


# ═══════════════════════════════════════════════════════
# BFS — Búsqueda en Anchura
# ═══════════════════════════════════════════════════════

def bfs(grid: Grid, diagonal: bool = True):
    """
    BFS — No pondera costes. Óptimo sólo si todos los costes son iguales.
    Muy rápido; útil como línea base.
    """
    start, end = grid.start, grid.end
    if start is None or end is None:
        return None, 0, 0, set()

    queue     = deque([start])
    came_from = {start: None}
    visited   = set([start])
    nodes_exp = 0
    t0        = time.perf_counter()

    while queue:
        current = queue.popleft()
        nodes_exp += 1

        if current == end:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            return path, nodes_exp, (time.perf_counter()-t0)*1000, visited

        for (nr, nc), _ in grid.get_neighbors(*current, diagonal=diagonal):
            neighbor = (nr, nc)
            if neighbor not in visited:
                visited.add(neighbor)
                came_from[neighbor] = current
                queue.append(neighbor)

    return None, nodes_exp, (time.perf_counter()-t0)*1000, visited


# ═══════════════════════════════════════════════════════
# GREEDY BEST-FIRST SEARCH
# ═══════════════════════════════════════════════════════

def greedy_bfs(grid: Grid, heuristic=h_octile, diagonal: bool = True):
    """
    Greedy — Expande siempre el nodo más prometedor según la heurística.
    Muy rápido pero NO garantiza la ruta óptima.
    """
    start, end = grid.start, grid.end
    if start is None or end is None:
        return None, 0, 0, set()

    open_heap  = [(heuristic(start, end), start)]
    came_from  = {start: None}
    visited    = set()
    nodes_exp  = 0
    t0         = time.perf_counter()

    while open_heap:
        _, current = heapq.heappop(open_heap)
        if current in visited:
            continue
        visited.add(current)
        nodes_exp += 1

        if current == end:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            return path, nodes_exp, (time.perf_counter()-t0)*1000, visited

        for (nr, nc), _ in grid.get_neighbors(*current, diagonal=diagonal):
            neighbor = (nr, nc)
            if neighbor not in visited:
                came_from[neighbor] = current
                heapq.heappush(open_heap, (heuristic(neighbor, end), neighbor))

    return None, nodes_exp, (time.perf_counter()-t0)*1000, visited


print("✅ Algoritmos definidos: A*, Dijkstra, BFS, Greedy BFS")

## 🎨 Sistema de Visualización

In [ ]:
# ═══════════════════════════════════════════════════════
# FUNCIÓN DE VISUALIZACIÓN
# ═══════════════════════════════════════════════════════

def visualize(grid: Grid,
              path=None,
              visited_nodes=None,
              title: str = "Mapa de Navegación",
              ax=None,
              show_stats: bool = True):
    """
    Renderiza el mapa con la ruta y los nodos explorados.

    Parámetros
    ----------
    grid          : Grid a visualizar
    path          : lista de (row, col) — ruta encontrada
    visited_nodes : set de (row, col) — nodos explorados
    title         : título del gráfico
    ax            : Axes de matplotlib (None → crea figura nueva)
    show_stats    : imprimir estadísticas en consola
    """
    display = grid.grid.copy()

    # Pintar nodos visitados
    if visited_nodes:
        for r, c in visited_nodes:
            if display[r, c] not in (START, END, WALL, SLOW, FAST):
                display[r, c] = VISITED

    # Pintar ruta
    if path:
        for r, c in path:
            if display[r, c] not in (START, END):
                display[r, c] = PATH

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(max(8, grid.cols*0.6),
                                        max(8, grid.rows*0.6)))

    ax.imshow(display, cmap=CMAP, norm=NORM,
              interpolation='nearest', aspect='equal')

    # Líneas de cuadrícula
    ax.set_xticks(np.arange(-0.5, grid.cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.rows, 1), minor=True)
    ax.grid(which='minor', color='#AAAAAA', linewidth=0.5)
    ax.tick_params(which='minor', size=0)
    ax.set_xticks(range(grid.cols));  ax.set_xticklabels(range(grid.cols), fontsize=7)
    ax.set_yticks(range(grid.rows));  ax.set_yticklabels(range(grid.rows), fontsize=7)

    # Marcadores de inicio / fin
    if grid.start:
        ax.text(grid.start[1], grid.start[0], '🚗',
                ha='center', va='center', fontsize=14)
    if grid.end:
        ax.text(grid.end[1], grid.end[0], '🏁',
                ha='center', va='center', fontsize=14)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)

    # Leyenda
    legend = [
        mpatches.Patch(color='#FFFFFF', label='Calle (coste 1)'),
        mpatches.Patch(color='#2C2C2C', label='Bloqueado / Edificio'),
        mpatches.Patch(color='#00CC44', label='Inicio 🚗'),
        mpatches.Patch(color='#FF4444', label='Destino 🏁'),
        mpatches.Patch(color='#3399FF', label='Ruta óptima'),
        mpatches.Patch(color='#FFE066', label='Nodos explorados'),
        mpatches.Patch(color='#FF9933', label='Zona lenta (coste 3)'),
        mpatches.Patch(color='#99FFCC', label='Zona rápida (coste 0.5)'),
    ]
    ax.legend(handles=legend, loc='upper right',
              fontsize=7, framealpha=0.92, edgecolor='gray')

    if show_stats and path:
        cost = grid.path_cost(path)
        info = (f"Pasos: {len(path)}  |  Coste: {cost:.2f}  |  "
                f"Explorados: {len(visited_nodes) if visited_nodes else '—'}")
        ax.set_xlabel(info, fontsize=9, labelpad=6)

    if standalone:
        plt.tight_layout()
        plt.show()


print("✅ Función de visualización lista.")

## 🖥️ Interfaz Interactiva

Ejecuta la siguiente celda para configurar tu propio mapa y ruta de forma interactiva.  
El programa te pedirá:
1. Dimensiones del mapa
2. Coordenadas de celdas bloqueadas (edificios / calles cortadas)
3. Zonas lentas y rápidas opcionales
4. Posición de inicio y destino
5. Si deseas movimiento diagonal

In [ ]:
# ═══════════════════════════════════════════════════════
# INTERFAZ INTERACTIVA
# ═══════════════════════════════════════════════════════

def _parse_coord(text):
    """Parsea 'fila col' o 'fila,col' → (int, int)"""
    text = text.replace(',', ' ')
    parts = text.split()
    return int(parts[0]), int(parts[1])

def _input_cells(grid, cell_type, prompt_label, color_hint):
    """Bucle de entrada de celdas de un tipo dado."""
    print(f"\n  [{color_hint}] {prompt_label}")
    print("  Introduce pares 'fila col' (uno por línea). Escribe 'listo' para continuar.")
    while True:
        raw = input("    > ").strip().lower()
        if raw in ('listo', 'ok', '', 'fin', 'done', 'no'):
            break
        try:
            r, c = _parse_coord(raw)
            if grid.set_cell(r, c, cell_type):
                print(f"    ✓  ({r},{c}) marcada")
        except Exception:
            print("    ✗  Formato inválido — usa: fila col")


def interactive_session():
    print("╔══════════════════════════════════════════════════════╗")
    print("║   SISTEMA DE NAVEGACIÓN AUTÓNOMA — Algoritmo A*     ║")
    print("╚══════════════════════════════════════════════════════╝")

    # ── 1. Dimensiones ──────────────────────────────────────
    print("\n📐 PASO 1 — Dimensiones del mapa")
    try:
        rows = int(input("  Filas   (recomendado 10–25): ").strip())
        cols = int(input("  Columnas(recomendado 10–25): ").strip())
        rows = max(3, min(rows, 40))
        cols = max(3, min(cols, 40))
    except ValueError:
        print("  ⚠️  Valores inválidos — usando 15×15 por defecto")
        rows, cols = 15, 15

    g = Grid(rows, cols)
    print(f"  ✅ Mapa de {rows}×{cols} creado.")

    # ── 2. Zonas bloqueadas ─────────────────────────────────
    print("\n🧱 PASO 2 — Edificios / Zonas bloqueadas")
    _input_cells(g, WALL, "Celdas BLOQUEADAS (edificios, calles cortadas)", "GRIS")

    # ── 3. Zonas especiales (opcional) ──────────────────────
    print("\n🟠 PASO 3 — Zonas especiales (opcional, escribe 'listo' para saltar)")
    _input_cells(g, SLOW, "Celdas LENTAS — obras, atascos (coste ×3)", "NARANJA")
    _input_cells(g, FAST, "Celdas RÁPIDAS — autopista (coste ×0.5)", "VERDE CLARO")

    # ── 4. Inicio ────────────────────────────────────────────
    print("\n🚗 PASO 4 — Posición de INICIO del vehículo")
    while True:
        try:
            r, c = _parse_coord(input("  Inicio (fila col): ").strip())
            if g.set_cell(r, c, START):
                break
        except Exception:
            print("  ✗  Formato inválido — usa: fila col")

    # ── 5. Destino ───────────────────────────────────────────
    print("\n🏁 PASO 5 — Posición de DESTINO")
    while True:
        try:
            r, c = _parse_coord(input("  Destino (fila col): ").strip())
            if g.set_cell(r, c, END):
                break
        except Exception:
            print("  ✗  Formato inválido — usa: fila col")

    # ── 6. Diagonal ──────────────────────────────────────────
    diag_resp = input("\n↗️  ¿Permitir movimiento diagonal? (s/n): ").strip().lower()
    diagonal  = diag_resp in ('s', 'si', 'sí', 'yes', 'y', '1')

    # ── Ejecución de A* ─────────────────────────────────────
    print("\n🔍 Ejecutando A*…")
    path, explored, ms, visited = astar(g, heuristic=h_octile, diagonal=diagonal)

    print("\n" + "═"*54)
    if path:
        cost = g.path_cost(path)
        print(f"  ✅ ¡RUTA ENCONTRADA!")
        print(f"     Pasos       : {len(path)}")
        print(f"     Coste total : {cost:.2f}")
        print(f"     Nodos expl. : {explored}")
        print(f"     Tiempo      : {ms:.3f} ms")
    else:
        print("  ❌ No se encontró ruta.")
        print("     Verifica que el destino sea accesible.")
    print("═"*54)

    title = (f"A* — {'Con diagonal' if diagonal else 'Sin diagonal'} "
             f"| Ruta: {len(path) if path else '—'} pasos")
    visualize(g, path=path, visited_nodes=visited, title=title)
    return g, path


# ── LANZAR SESIÓN INTERACTIVA ───────────────────────────
g_interactive, path_interactive = interactive_session()

## 🗺️ Demo Automática — Mapa de Ciudad Predefinido

Ejecuta esta celda si prefieres ver directamente un ejemplo completo sin introducir datos manualmente.

In [ ]:
# ═══════════════════════════════════════════════════════
# MAPA DE CIUDAD PREDEFINIDO — Demo automática
# ═══════════════════════════════════════════════════════

def build_demo_city(rows=15, cols=15):
    """
    Genera un mapa urbano de demostración con:
    - Bloques de edificios (paredes)
    - Una zona de obras (lenta)
    - Una avenida principal (rápida)
    """
    g = Grid(rows, cols)

    # Bloque norte
    g.set_walls_from_list([(1,2),(1,3),(1,4),(2,2),(2,3),(2,4),(3,2),(3,3)])
    # Bloque central-izquierda
    g.set_walls_from_list([(5,1),(5,2),(6,1),(6,2),(7,1),(7,2)])
    # Bloque central
    g.set_walls_from_list([(5,6),(5,7),(5,8),(6,6),(6,7),(6,8),
                            (7,6),(7,7),(8,6),(8,7)])
    # Bloque central-derecha
    g.set_walls_from_list([(4,11),(4,12),(5,11),(5,12),(6,11),(6,12)])
    # Bloque sur
    g.set_walls_from_list([(10,3),(10,4),(10,5),(11,3),(11,4),(11,5),
                             (12,3),(12,4)])
    # Bloque sur-derecha
    g.set_walls_from_list([(9,9),(9,10),(10,9),(10,10),(11,9),(11,10)])

    # Zona de obras (lenta)
    for c in range(0, 6):
        g.set_cell(9, c, SLOW)

    # Avenida principal (rápida) — columna central
    for r in range(0, rows):
        if g.grid[r, 13] == EMPTY:
            g.set_cell(r, 13, FAST)

    # Inicio y destino
    g.set_cell(0, 0, START)
    g.set_cell(14, 14, END)

    return g

# Construir y visualizar el mapa vacío
g_demo = build_demo_city()
print("📍 Mapa de ciudad generado")
print(f"   Inicio : {g_demo.start}")
print(f"   Destino: {g_demo.end}")

visualize(g_demo, title="Mapa Urbano de Demostración — Sin ruta")

## 🔬 Comparativa de Heurísticas sobre A*

In [ ]:
# ═══════════════════════════════════════════════════════
# A* CON DIFERENTES HEURÍSTICAS — comparativa visual
# ═══════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Comparativa de Heurísticas — Algoritmo A*\n(Mapa Urbano Demo, movimiento diagonal)",
             fontsize=15, fontweight='bold')

resultados_heuristicas = {}

for ax, (h_name, h_fn) in zip(axes.flatten(), HEURISTICS.items()):
    g_test = build_demo_city()
    path, explored, ms, visited = astar(g_test, heuristic=h_fn, diagonal=True)
    coste_val = g_test.path_cost(path) if path else float('inf')
    resultados_heuristicas[h_name] = {
        'pasos': len(path) if path else 0,
        'coste': coste_val,
        'explorados': explored,
        'ms': ms
    }
    coste_str = f"{coste_val:.2f}" if path else "—"
    pasos_str = str(len(path)) if path else "—"
    title = (f"A* — {h_name}\n"
             f"Pasos: {pasos_str} | "
             f"Coste: {coste_str} | "
             f"Explorados: {explored}")
    visualize(g_test, path=path, visited_nodes=visited, title=title, ax=ax, show_stats=False)

plt.tight_layout()
plt.show()

print("\n📊 Resumen de heurísticas:")
print(f"{'Heurística':<12} {'Pasos':>7} {'Coste':>8} {'Explorados':>12} {'Tiempo(ms)':>12}")
print("─" * 55)
for name, res in resultados_heuristicas.items():
    print(f"{name:<12} {res['pasos']:>7} {res['coste']:>8.2f} "
          f"{res['explorados']:>12} {res['ms']:>11.3f}")


## 📊 Comparativa de Algoritmos

Comparamos A*, Dijkstra, BFS y Greedy Best-First sobre el mismo mapa.

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPARATIVA DE ALGORITMOS — mismo mapa, misma ruta
# ═══════════════════════════════════════════════════════

algoritmos = {
    'A* (Octile)'   : lambda g: astar(g,      heuristic=h_octile, diagonal=True),
    'Dijkstra'      : lambda g: dijkstra(g,                        diagonal=True),
    'BFS'           : lambda g: bfs(g,                             diagonal=True),
    'Greedy BFS'    : lambda g: greedy_bfs(g,  heuristic=h_octile, diagonal=True),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Comparativa de Algoritmos de Búsqueda de Rutas\n"
             "(Mapa Urbano Demo — Movimiento diagonal)",
             fontsize=15, fontweight='bold')

resultados_alg = {}

for ax, (alg_name, alg_fn) in zip(axes.flatten(), algoritmos.items()):
    g_test = build_demo_city()
    path, explored, ms, visited = alg_fn(g_test)
    cost = g_test.path_cost(path) if path else float('inf')
    resultados_alg[alg_name] = {
        'pasos': len(path) if path else 0,
        'coste': cost,
        'explorados': explored,
        'ms': ms,
        'optimo': path is not None
    }
    cost_str  = f"{cost:.2f}" if path else "—"
    pasos_str = str(len(path)) if path else "—"
    title = (f"{alg_name}\n"
             f"Pasos: {pasos_str} | "
             f"Coste: {cost_str} | "
             f"Explorados: {explored}")
    visualize(g_test, path=path, visited_nodes=visited, title=title, ax=ax, show_stats=False)

plt.tight_layout()
plt.show()

# ── Tabla comparativa ───────────────────────────────────
print("\n" + "═"*65)
print("  TABLA COMPARATIVA DE ALGORITMOS")
print("═"*65)
print(f"{'Algoritmo':<16} {'Pasos':>6} {'Coste':>8} {'Explorados':>12} {'Tiempo(ms)':>12}")
print("─"*65)
for name, res in resultados_alg.items():
    marker = " ✅" if res['coste'] == min(r['coste'] for r in resultados_alg.values()) else ""
    print(f"{name:<16} {res['pasos']:>6} {res['coste']:>8.2f} "
          f"{res['explorados']:>12} {res['ms']:>11.3f}{marker}")
print("═"*65)
print("\n  ✅ = Ruta de menor coste encontrada")


## 📈 Gráficas de Rendimiento Comparativo

In [ ]:
# ═══════════════════════════════════════════════════════
# BENCHMARK — Rendimiento en función del tamaño del mapa
# ═══════════════════════════════════════════════════════

def benchmark_grid(size):
    """Genera un mapa cuadrado con paredes aleatorias distribuidas uniformemente."""
    import random
    random.seed(42)
    g = Grid(size, size)
    # ~20% de obstáculos
    for r in range(size):
        for c in range(size):
            if random.random() < 0.20 and (r,c) not in ((0,0),(size-1,size-1)):
                g.grid[r][c] = WALL
    g.set_cell(0, 0, START)
    g.set_cell(size-1, size-1, END)
    return g

sizes = [10, 15, 20, 25, 30]
bench_results = {name: {'explored': [], 'ms': [], 'cost': []}
                 for name in algoritmos}

print("⏳ Ejecutando benchmark…")
for sz in sizes:
    g_bench = benchmark_grid(sz)
    for alg_name, alg_fn in algoritmos.items():
        path, explored, ms, _ = alg_fn(g_bench.copy())
        cost = g_bench.path_cost(path) if path else 0
        bench_results[alg_name]['explored'].append(explored)
        bench_results[alg_name]['ms'].append(ms)
        bench_results[alg_name]['cost'].append(cost)
    print(f"  ✓ Tamaño {sz}×{sz}")

print("✅ Benchmark completado.\n")

# ── Gráficas ─────────────────────────────────────────────
colors_bench = ['#3399FF', '#FF6633', '#AA44FF', '#33CC99']
markers      = ['o', 's', '^', 'D']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Benchmark: Rendimiento vs Tamaño del Mapa",
             fontsize=14, fontweight='bold')

for (alg_name, res), col, mk in zip(bench_results.items(), colors_bench, markers):
    axes[0].plot(sizes, res['explored'], marker=mk, color=col, label=alg_name, linewidth=2)
    axes[1].plot(sizes, res['ms'],       marker=mk, color=col, label=alg_name, linewidth=2)
    axes[2].plot(sizes, res['cost'],     marker=mk, color=col, label=alg_name, linewidth=2)

for ax, ylabel, title in zip(axes,
    ['Nodos explorados', 'Tiempo (ms)', 'Coste de ruta'],
    ['Nodos Explorados vs Tamaño', 'Tiempo de Ejecución vs Tamaño', 'Calidad de Ruta vs Tamaño']):
    ax.set_xlabel('Tamaño del mapa (N×N)', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(sizes)
    ax.set_xticklabels([f'{s}×{s}' for s in sizes], rotation=30, ha='right')

plt.tight_layout()
plt.show()

## ↗️ Comparativa: Movimiento Cardinal vs. Diagonal

In [ ]:
# ═══════════════════════════════════════════════════════
# EFECTO DEL MOVIMIENTO DIAGONAL
# ═══════════════════════════════════════════════════════

g_base = build_demo_city()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle("Impacto del Movimiento Diagonal en A*",
             fontsize=14, fontweight='bold')

# Sin diagonal (Manhattan)
g1 = g_base.copy()
path1, exp1, ms1, vis1 = astar(g1, heuristic=h_manhattan, diagonal=False)
cost1 = g1.path_cost(path1) if path1 else float('inf')
visualize(g1, path=path1, visited_nodes=vis1,
          title=f"Sin Diagonal (Manhattan)\nPasos:{len(path1) if path1 else '—'} "
                f"Coste:{cost1:.2f} Exp:{exp1}",
          ax=ax1, show_stats=False)

# Con diagonal (Octile)
g2 = g_base.copy()
path2, exp2, ms2, vis2 = astar(g2, heuristic=h_octile, diagonal=True)
cost2 = g2.path_cost(path2) if path2 else float('inf')
visualize(g2, path=path2, visited_nodes=vis2,
          title=f"Con Diagonal (Octile)\nPasos:{len(path2) if path2 else '—'} "
                f"Coste:{cost2:.2f} Exp:{exp2}",
          ax=ax2, show_stats=False)

plt.tight_layout()
plt.show()

mejora_exp   = (exp1 - exp2) / exp1 * 100 if exp1 > 0 else 0
mejora_cost  = (cost1 - cost2) / cost1 * 100 if cost1 > 0 else 0
print(f"\n  Reducción de nodos explorados : {mejora_exp:+.1f}%")
print(f"  Mejora en coste de ruta        : {mejora_cost:+.1f}%")

## 📝 Conclusiones

### Análisis de Resultados

#### Heurísticas en A*
- **Octile** es la heurística más eficiente para movimiento en 8 direcciones: minimiza los nodos explorados a la vez que garantiza la optimalidad.
- **Manhattan** resulta inadecuada para movimiento diagonal (subestima el coste real), generando más exploración innecesaria.
- **Euclídea** es admisible pero menos precisa que Octile para cuadrículas con costes √2.
- **Chebyshev** funciona bien cuando el coste diagonal es exactamente 1 (no el caso de este sistema).

#### Comparativa de Algoritmos
| Criterio | A* | Dijkstra | BFS | Greedy BFS |
|---|---|---|---|---|
| **Garantía de óptimo** | ✅ | ✅ | Solo coste uniforme | ❌ |
| **Velocidad** | ⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Uso de memoria** | Medio | Alto | Alto | Bajo |
| **Soporte costes** | ✅ | ✅ | ❌ | ❌ |
| **Idoneidad navegación** | ✅✅ | ✅ | Limitado | No recomendado |

#### Movimiento Diagonal
El soporte diagonal reduce notablemente el coste de ruta y los nodos explorados, especialmente en mapas con pocos obstáculos. La prevención de *corner cutting* es esencial para garantizar trayectorias físicamente válidas.

### Conclusión Final

**A* con heurística Octile** es la elección óptima para un sistema de navegación autónoma urbana:
1. Garantiza la ruta de mínimo coste.
2. Explora significativamente menos nodos que Dijkstra o BFS.
3. Se adapta a terrenos de coste variable (zonas lentas, rápidas).
4. Soporta movimiento diagonal realista.
5. Escala bien con mapas grandes.

Para futuras extensiones se podría implementar **Theta*** (sucesor de A* con movimiento en ángulo libre) o **Jump Point Search (JPS)** para mapas de coste uniforme, con ganancias de velocidad de hasta 10×.

---
*Proyecto desarrollado como prototipo interno — Departamento de IA Automovilística*